# TFE — Comparativa de RAG, fine-tuning y function calling
**Cuaderno de experimentos (Google Colab Pro · GPU L4)**
Hernán David Mejía Martínez · Máster Universitario en Inteligencia Artificial (UNIR)

Este cuaderno implementa el protocolo del capítulo 4 de la memoria. Está pensado para **Colab Pro sin ejecución en segundo plano**:

* Cada **etapa** es independiente y **reanudable**: escribe sus resultados de forma incremental en Google Drive y, si se vuelve a ejecutar, salta lo ya hecho.
* El **modelo generador** (Llama 3.1 8B Instruct) y el **modelo juez** (Qwen2.5-14B-Instruct-AWQ) no caben a la vez en la L4: se usan en etapas distintas. Entre etapas de GPU conviene **reiniciar el entorno** (Entorno de ejecución → Reiniciar sesión) y volver a ejecutar solo las celdas de la sección 0.
* Cada etapa registra su duración y una estimación de **unidades de cómputo** en `budget.jsonl`.

| Etapa | GPU | Qué hace | Coste aprox. (L4) |
|---|---|---|---|
| 0 | — | Instalación, configuración, utilidades | — |
| 1 | — | Subconjuntos FRAMES (400) y HotpotQA (500) | CPU |
| 2 | — | Corpus K (Wikipedia con revid; contextos HotpotQA), troceado | CPU |
| 3 | L4 | Índice BGE-M3 + FAISS, recuperación RAG y evidencia del oráculo | 0,5 h |
| 4 | L4 | Datos sintéticos para el fine-tuning (vLLM) | 2 h |
| 5 | L4 | Entrenamiento QLoRA (2 benchmarks × 3 semillas) | 14 h |
| 6 | L4 | Inferencia de las 5 configuraciones × 3 repeticiones | 4 h |
| 7 | L4 | Latencia sin lotes (100 preguntas) | 1,5 h |
| 8 | L4 | Juez: exactitud FRAMES, F_ref, F_ctx (con deduplicación) | 7 h |
| 9 | — | Validación del juez (kappa) y análisis estadístico | CPU |
| 10 | L4 | Bloque complementario C2 (StableToolBench, opcional) | 5 h |

**Modo piloto:** con `PILOTO = True` todas las etapas trabajan sobre 50 preguntas por benchmark (fase 3 del plan). Cambiar a `False` para la ejecución completa **en un directorio distinto** (se hace automáticamente).

**Requisitos previos:** (1) aceptar la licencia de `meta-llama/Llama-3.1-8B-Instruct` en Hugging Face; (2) guardar el token en los *Secretos* de Colab con el nombre `HF_TOKEN`; (3) opcional: `WANDB_API_KEY`; (4) elegir *Entorno de ejecución → Cambiar tipo → GPU L4*.

## 0. Entorno, configuración y utilidades
Ejecutar siempre al iniciar o reiniciar la sesión.

In [ ]:
# 0.1 Montar Drive y comprobar GPU
import os, sys, json, time, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'Sin GPU (etapas CPU solamente)')

In [ ]:
# 0.2 Instalación (solo la primera vez en cada sesión; ~5 min)
# Tras el piloto, fija aquí las versiones exactas que funcionaron (pip freeze queda en el manifiesto).
!pip -q install "vllm" "transformers" "peft" "trl" "bitsandbytes" "accelerate" "datasets" \
    "openai" "sentence-transformers" "faiss-cpu" "statsmodels" "scikit-learn" "wandb" "requests" "tqdm"
# Si Colab lo pide tras instalar, reinicia la sesión y continúa desde la celda 0.3.

In [ ]:
# 0.3 Configuración fija del experimento (capítulo 4). No cambiar tras el piloto salvo motivo documentado.
from dataclasses import dataclass, field, asdict

PILOTO = True

@dataclass
class Config:
    # rutas
    drive_root: str = '/content/drive/MyDrive/TFE_comparativa'
    # modelos
    generador: str = 'meta-llama/Llama-3.1-8B-Instruct'
    juez: str = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
    embeddings: str = 'BAAI/bge-m3'
    # subconjuntos
    n_frames: int = 400
    n_hotpot: int = 500
    n_piloto: int = 50
    semilla_muestreo: int = 2026
    # corpus y recuperación
    max_tokens_articulo: int = 3072      # truncado FRAMES (4.3.1)
    chunk_tokens: int = 256
    chunk_solape: int = 32
    k_rag: int = 6                       # presupuesto de evidencia (4.2)
    fc_max_llamadas: int = 3
    k_fc: int = 2
    k_verificacion_frames: int = 8       # fragmentos de artículos de referencia usados por el juez
    # generación
    temperatura: float = 0.0
    max_tokens_salida: int = 256
    max_model_len: int = 8192
    gpu_mem_util: float = 0.90
    repeticiones: tuple = (0, 1, 2)
    concurrencia: int = 16
    # fine-tuning (4.3.4)
    semillas_ft: tuple = (13, 42, 2026)
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lr: float = 2e-4
    epocas: int = 2
    max_seq_len: int = 1024
    batch_size: int = 4
    grad_accum: int = 4
    qa_por_fragmento: int = 2
    umbral_fuga: float = 0.90            # similitud coseno máxima con preguntas evaluadas
    # latencia y juez
    n_latencia: int = 100
    kappa_minimo: float = 0.60
    n_anotacion: int = 100
    # análisis
    bootstrap: int = 10000
    umbral_pp: float = 0.05              # relevancia práctica precisión/fidelidad
    umbral_rel_coste: float = 0.20       # relevancia práctica latencia/coste
    # tarifas de Colab (unidades/hora, marzo de 2026)
    uc_hora: dict = field(default_factory=lambda: {'L4': 1.71, 'A100': 5.40, 'T4': 1.19, 'CPU': 0.08})

CFG = Config()
RUN = 'piloto' if PILOTO else 'completo'
ROOT = f'{CFG.drive_root}/{RUN}'
DIRS = {k: f'{ROOT}/{k}' for k in ['datos', 'corpus', 'indice', 'ft', 'adaptadores', 'salidas', 'juez', 'analisis', 'logs']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
CONFIGS = ['B0', 'RAG', 'FT', 'FC', 'OR']
BENCHS = ['hotpotqa', 'frames']
print(RUN, ROOT)

In [ ]:
# 0.4 Utilidades: JSONL incremental, reanudación, registro de coste y manifiesto
import hashlib, re, random, math, contextlib, datetime as dt
import numpy as np

def jl_append(path, rows):
    with open(path, 'a', encoding='utf8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def jl_read(path):
    if not os.path.exists(path):
        return []
    out = []
    with open(path, encoding='utf8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass  # línea truncada por una desconexión: se ignora y se rehará
    return out

def done_keys(path, keyf):
    return {keyf(r) for r in jl_read(path)}

def sha(*parts):
    return hashlib.sha1('␟'.join(map(str, parts)).encode('utf8')).hexdigest()[:16]

def gpu_name():
    try:
        n = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout
    except FileNotFoundError:
        return 'CPU'
    for k in ['L4', 'A100', 'T4']:
        if k in n:
            return k
    return 'CPU' if not n.strip() else n.strip()

@contextlib.contextmanager
def etapa(nombre):
    t0 = time.time(); print(f'▶ {nombre}')
    try:
        yield
    finally:
        h = (time.time() - t0) / 3600; g = gpu_name()
        uc = h * CFG.uc_hora.get(g, CFG.uc_hora['L4'])
        jl_append(f"{DIRS['logs']}/budget.jsonl", [{'etapa': nombre, 'gpu': g, 'horas': round(h, 4), 'uc_estimadas': round(uc, 3),
                                                   'fin': dt.datetime.now().isoformat(timespec='seconds')}])
        print(f'■ {nombre}: {h*60:.1f} min · ~{uc:.2f} UC ({g})')

def manifiesto(extra=None):
    m = {'cfg': asdict(CFG), 'run': RUN, 'gpu': gpu_name(), 'fecha': dt.datetime.now().isoformat(timespec='seconds'), 'python': sys.version}
    try:
        m['pip_freeze'] = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.splitlines()
    except Exception:
        pass
    if extra:
        m.update(extra)
    jl_append(f"{DIRS['logs']}/manifiesto.jsonl", [m])

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

In [ ]:
# 0.5 Secretos y tokenizador del generador (para contar tokens y trocear)
if IN_COLAB:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    with contextlib.suppress(Exception):
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
from transformers import AutoTokenizer
TOK = AutoTokenizer.from_pretrained(CFG.generador, token=os.environ.get('HF_TOKEN'))
if TOK.pad_token is None:
    TOK.pad_token = TOK.eos_token
manifiesto()

### 0.6 Instrucciones, formato de salida y métricas
Comunes a todas las configuraciones (homogeneidad, Tabla 6).

In [ ]:
SYSTEM = (
    "Eres un asistente que responde preguntas de conocimiento. Responde SIEMPRE con un objeto JSON válido "
    "con dos campos: \"respuesta\" (la respuesta más breve posible: una entidad, fecha, número o frase corta) y "
    "\"justificacion\" (una o dos oraciones con los hechos en los que se basa la respuesta). No añadas texto fuera del JSON."
)
SYSTEM_CONTEXTO = SYSTEM + " Utiliza únicamente la información de los fragmentos proporcionados."
SYSTEM_FC = SYSTEM + (" Dispones de la herramienta buscar_corpus para consultar la base de conocimiento. "
                      f"Puedes llamarla hasta {CFG.fc_max_llamadas} veces; reformula la consulta si los resultados no bastan. "
                      "Cuando tengas suficiente información, responde con el JSON final.")

TOOL_BUSCAR = [{
    'type': 'function',
    'function': {
        'name': 'buscar_corpus',
        'description': 'Busca en la base de conocimiento y devuelve los fragmentos de texto más relevantes para la consulta.',
        'parameters': {'type': 'object',
                       'properties': {'consulta': {'type': 'string', 'description': 'Consulta de búsqueda en lenguaje natural.'}},
                       'required': ['consulta']},
    },
}]

def fmt_fragmentos(frs):
    return '\n\n'.join(f"[{i+1}] ({f['titulo']}) {f['texto']}" for i, f in enumerate(frs))

def mensajes(config, pregunta, frags=None):
    if config in ('B0', 'FT'):
        return [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': f'Pregunta: {pregunta}'}]
    if config in ('RAG', 'OR'):
        return [{'role': 'system', 'content': SYSTEM_CONTEXTO},
                {'role': 'user', 'content': f'Fragmentos:\n{fmt_fragmentos(frags)}\n\nPregunta: {pregunta}'}]
    if config == 'FC':
        return [{'role': 'system', 'content': SYSTEM_FC}, {'role': 'user', 'content': f'Pregunta: {pregunta}'}]
    raise ValueError(config)

def parse_salida(txt):
    # Extrae {respuesta, justificacion}; marca formato_valido=False si no hay JSON utilizable.
    txt = (txt or '').strip()
    cand = re.findall(r'\{.*\}', txt, flags=re.S)
    for c in ([txt] + cand):
        try:
            o = json.loads(c)
            if isinstance(o, dict) and 'respuesta' in o:
                return {'respuesta': str(o.get('respuesta', '')).strip(), 'justificacion': str(o.get('justificacion', '')).strip(), 'formato_valido': True}
        except Exception:
            continue
    return {'respuesta': txt.split('\n')[0][:200], 'justificacion': '', 'formato_valido': False}

# --- Métricas oficiales de HotpotQA (normalización estándar)
import string
from collections import Counter
def normalize_answer(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    p, g = normalize_answer(pred), normalize_answer(gold)
    if p in ('yes', 'no', 'noanswer') or g in ('yes', 'no', 'noanswer'):
        if p != g:
            return 0.0
    pt, gt = p.split(), g.split()
    common = Counter(pt) & Counter(gt); ns = sum(common.values())
    if ns == 0:
        return 0.0
    prec, rec = ns / len(pt), ns / len(gt)
    return 2 * prec * rec / (prec + rec)

In [ ]:
# 0.7 Troceado por tokens (256 con solape 32) con el tokenizador del generador
def trocear(texto, n=None, solape=None, tok=None):
    tok = tok or TOK
    n = n or CFG.chunk_tokens; solape = CFG.chunk_solape if solape is None else solape
    ids = tok.encode(texto, add_special_tokens=False)
    out, i = [], 0
    while i < len(ids):
        out.append(tok.decode(ids[i:i + n]))
        if i + n >= len(ids):
            break
        i += n - solape
    return out

def truncar(texto, max_tokens, tok=None):
    tok = tok or TOK
    ids = tok.encode(texto, add_special_tokens=False)
    return tok.decode(ids[:max_tokens]), len(ids)

## 1. Subconjuntos de evaluación (CPU)
Muestreo estratificado con semilla fija: FRAMES por tipo de razonamiento (primera etiqueta), HotpotQA (validación, configuración *distractor*) por tipo de pregunta. Los identificadores se guardan y no vuelven a muestrearse.

In [ ]:
import ast
from datasets import load_dataset

def estratificado(items, clave, n, semilla):
    rng = random.Random(semilla)
    grupos = {}
    for it in items:
        grupos.setdefault(clave(it), []).append(it)
    tot = len(items); sel = []
    for g, lst in sorted(grupos.items()):
        rng.shuffle(lst)
        sel += lst[:max(1, round(n * len(lst) / tot))]
    rng.shuffle(sel)
    return sel[:n]

def preparar_subconjuntos():
    pf = f"{DIRS['datos']}/preguntas.jsonl"
    if os.path.exists(pf):
        print('Ya existen:', pf); return
    fr = load_dataset('google/frames-benchmark', split='test')
    frames = []
    for i, r in enumerate(fr):
        links = ast.literal_eval(r['wiki_links']) if isinstance(r['wiki_links'], str) else r['wiki_links']
        frames.append({'bench': 'frames', 'qid': f'frames_{i}', 'pregunta': r['Prompt'], 'respuesta': str(r['Answer']),
                       'tipo': (r.get('reasoning_types') or 'otro').split('|')[0].strip(), 'wiki_links': links})
    try:
        hp = load_dataset('hotpotqa/hotpot_qa', 'distractor', split='validation')
    except Exception:
        hp = load_dataset('hotpot_qa', 'distractor', split='validation')
    hotpot = []
    for r in hp:
        ctx = [{'titulo': t, 'oraciones': s} for t, s in zip(r['context']['title'], r['context']['sentences'])]
        sf = list(zip(r['supporting_facts']['title'], r['supporting_facts']['sent_id']))
        hotpot.append({'bench': 'hotpotqa', 'qid': r['id'], 'pregunta': r['question'], 'respuesta': r['answer'],
                       'tipo': r['type'], 'contexto': ctx, 'hechos_apoyo': sf})
    n_f = CFG.n_piloto if PILOTO else CFG.n_frames
    n_h = CFG.n_piloto if PILOTO else CFG.n_hotpot
    sel = estratificado(frames, lambda x: x['tipo'], n_f, CFG.semilla_muestreo) + \
          estratificado(hotpot, lambda x: x['tipo'], n_h, CFG.semilla_muestreo)
    jl_append(pf, sel)
    print({b: sum(1 for x in sel if x['bench'] == b) for b in BENCHS})

with etapa('1_subconjuntos'):
    preparar_subconjuntos()
PREG = {q['qid']: q for q in jl_read(f"{DIRS['datos']}/preguntas.jsonl")}

## 2. Corpus de conocimiento K y troceado (CPU)
* **HotpotQA:** unión de los párrafos (2 de apoyo + 8 distractores) de las preguntas seleccionadas; un fragmento por párrafo. La evidencia de referencia E\* son las oraciones de apoyo.
* **FRAMES:** artículos enlazados en las preguntas seleccionadas, descargados de la API de Wikipedia con su `revid`, truncados a 3 072 tokens y troceados (256/32). E\* son los artículos enlazados.

> Rellena `CONTACTO_UA` con un correo o URL de contacto: la API de Wikipedia exige un *User-Agent* identificable.

In [ ]:
import requests, urllib.parse
CONTACTO_UA = 'RELLENAR_CONTACTO'   # p. ej. URL del repositorio o correo institucional
UA = {'User-Agent': f'TFE-UNIR-comparativa/1.0 ({CONTACTO_UA})'}
WIKI_API = 'https://en.wikipedia.org/w/api.php'

def titulo_de_url(u):
    return urllib.parse.unquote(u.rstrip('/').split('/wiki/')[-1]).replace('_', ' ').split('#')[0]

def descargar_articulo(titulo, reintentos=4):
    params = {'action': 'query', 'prop': 'extracts|revisions', 'explaintext': 1, 'rvprop': 'ids|timestamp',
              'titles': titulo, 'redirects': 1, 'format': 'json', 'formatversion': 2}
    for a in range(reintentos):
        try:
            r = requests.get(WIKI_API, params=params, headers=UA, timeout=30)
            r.raise_for_status()
            p = r.json()['query']['pages'][0]
            if p.get('missing'):
                return None
            rev = p['revisions'][0]
            return {'titulo': p['title'], 'texto': p.get('extract', ''), 'revid': rev['revid'], 'rev_fecha': rev['timestamp']}
        except Exception as e:
            time.sleep(2 ** a)
    return None

def construir_corpus():
    art_path = f"{DIRS['corpus']}/frames_articulos.jsonl"
    hechos = done_keys(art_path, lambda r: r['url'])
    urls = sorted({u for q in PREG.values() if q['bench'] == 'frames' for u in q['wiki_links']})
    pend = [u for u in urls if u not in hechos]
    print(f'FRAMES: {len(urls)} artículos, pendientes {len(pend)}')
    from tqdm.auto import tqdm
    for u in tqdm(pend):
        a = descargar_articulo(titulo_de_url(u))
        jl_append(art_path, [{'url': u, **(a or {'titulo': titulo_de_url(u), 'texto': '', 'revid': None, 'rev_fecha': None}), 'ok': a is not None}])
        time.sleep(0.2)
    # Fragmentos
    fr_path = f"{DIRS['corpus']}/fragmentos.jsonl"
    if os.path.exists(fr_path):
        print('Fragmentos ya construidos'); return
    frs = []
    for a in jl_read(art_path):
        if not a['ok']:
            continue
        texto, n_orig = truncar(a['texto'], CFG.max_tokens_articulo)
        for j, t in enumerate(trocear(texto)):
            frs.append({'bench': 'frames', 'fid': f"fr_{sha(a['url'])}_{j}", 'url': a['url'], 'titulo': a['titulo'], 'texto': t})
    vistos = set()
    for q in PREG.values():
        if q['bench'] != 'hotpotqa':
            continue
        for c in q['contexto']:
            if c['titulo'] in vistos:
                continue
            vistos.add(c['titulo'])
            texto, _ = truncar(' '.join(c['oraciones']), CFG.chunk_tokens)
            frs.append({'bench': 'hotpotqa', 'fid': f"hp_{sha(c['titulo'])}", 'url': None, 'titulo': c['titulo'], 'texto': texto})
    jl_append(fr_path, frs)
    fallos = [a['url'] for a in jl_read(art_path) if not a['ok']]
    print({b: sum(1 for f in frs if f['bench'] == b) for b in BENCHS}, '| artículos no descargados:', len(fallos))

with etapa('2_corpus'):
    construir_corpus()
FRAG = {f['fid']: f for f in jl_read(f"{DIRS['corpus']}/fragmentos.jsonl")}

def evidencia_ref_hotpot(q):
    # Oraciones de apoyo anotadas (E* de HotpotQA)
    ctx = {c['titulo']: c['oraciones'] for c in q['contexto']}
    return [{'titulo': t, 'texto': ctx[t][i]} for t, i in q['hechos_apoyo'] if t in ctx and i < len(ctx[t])]

## 3. Índice vectorial y recuperación (GPU, ~0,5 h)
BGE-M3 (vectores densos normalizados) + FAISS exacto por producto interno, un índice por benchmark. Se precalculan: la recuperación de RAG (k = 6, consulta = pregunta), los fragmentos del oráculo (FRAMES: los 6 fragmentos de los artículos de referencia más similares a la pregunta; HotpotQA: los 2 párrafos de apoyo) y los vectores de las preguntas (para el filtro de fuga del fine-tuning).

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

def cargar_embedder(device=None):
    import torch
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    m = SentenceTransformer(CFG.embeddings, device=device)
    if device == 'cuda':
        m.half()
    return m

def embed(model, textos, bs=64):
    return model.encode(textos, batch_size=bs, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=len(textos) > 500).astype('float32')

def construir_indices():
    emb = cargar_embedder()
    for b in BENCHS:
        ip = f"{DIRS['indice']}/{b}.faiss"
        if os.path.exists(ip):
            print('Índice existente', b); continue
        fids = [k for k, f in FRAG.items() if f['bench'] == b]
        X = embed(emb, [f"{FRAG[k]['titulo']}. {FRAG[k]['texto']}" for k in fids])
        idx = faiss.IndexFlatIP(X.shape[1]); idx.add(X)
        faiss.write_index(idx, ip); json.dump(fids, open(f"{DIRS['indice']}/{b}_fids.json", 'w'))
        np.save(f"{DIRS['indice']}/{b}_X.npy", X)
    # recuperación RAG y oráculo
    rp = f"{DIRS['indice']}/recuperacion.jsonl"
    if os.path.exists(rp):
        print('Recuperación existente'); return
    rows = []
    for b in BENCHS:
        idx = faiss.read_index(f"{DIRS['indice']}/{b}.faiss"); fids = json.load(open(f"{DIRS['indice']}/{b}_fids.json"))
        X = np.load(f"{DIRS['indice']}/{b}_X.npy")
        qs = [q for q in PREG.values() if q['bench'] == b]
        Q = embed(emb, [q['pregunta'] for q in qs])
        np.save(f"{DIRS['indice']}/{b}_Q.npy", Q); json.dump([q['qid'] for q in qs], open(f"{DIRS['indice']}/{b}_qids.json", 'w'))
        _, I = idx.search(Q, CFG.k_rag)
        for q, qv, ii in zip(qs, Q, I):
            if b == 'hotpotqa':
                apoyo = {t for t, _ in q['hechos_apoyo']}
                oraculo = [f"hp_{sha(t)}" for t in apoyo if f"hp_{sha(t)}" in FRAG]
            else:
                gold = [j for j, f in enumerate(fids) if FRAG[f]['url'] in set(q['wiki_links'])]
                s = X[gold] @ qv
                oraculo = [fids[gold[j]] for j in np.argsort(-s)[:CFG.k_rag]]
            rows.append({'qid': q['qid'], 'rag': [fids[j] for j in ii], 'oraculo': oraculo})
    jl_append(rp, rows)

with etapa('3_indices'):
    construir_indices()
RECUP = {r['qid']: r for r in jl_read(f"{DIRS['indice']}/recuperacion.jsonl")}

In [ ]:
# Recuperador de consultas para function calling (en CPU para que RAG y FC midan la latencia en las mismas condiciones; con caché)
class Recuperador:
    def __init__(self, device='cpu'):
        self.emb = cargar_embedder(device)
        self.idx = {b: faiss.read_index(f"{DIRS['indice']}/{b}.faiss") for b in BENCHS}
        self.fids = {b: json.load(open(f"{DIRS['indice']}/{b}_fids.json")) for b in BENCHS}
        self.cache = {}
    def buscar(self, bench, consulta, k):
        key = (bench, consulta, k)
        if key not in self.cache:
            v = embed(self.emb, [consulta]); _, I = self.idx[bench].search(v, k)
            self.cache[key] = [self.fids[bench][j] for j in I[0]]
        return self.cache[key]

## Servidor vLLM (común a las etapas 4, 6, 7, 8 y 10)
Todas las configuraciones se sirven con el mismo servidor y la misma interfaz, para que la única diferencia sea la estrategia. El fine-tuning se sirve como adaptadores LoRA sobre los mismos pesos en bfloat16.

In [ ]:
import asyncio, glob
from openai import AsyncOpenAI
PORT = 8000
TEMPLATE_URL = 'https://raw.githubusercontent.com/vllm-project/vllm/main/examples/tool_chat_template_llama3.1_json.jinja'

def plantilla_tools():
    p = '/content/tool_chat_template_llama3.1_json.jinja'
    if not os.path.exists(p):
        r = requests.get(TEMPLATE_URL, timeout=30)
        if r.ok:
            open(p, 'w').write(r.text)
        else:
            return None
    return p

SERVER = {'proc': None, 'model': None}
def iniciar_servidor(modelo, loras=None, tools=False, extra=()):
    parar_servidor()
    cmd = ['python', '-m', 'vllm.entrypoints.openai.api_server', '--model', modelo, '--port', str(PORT),
           '--max-model-len', str(CFG.max_model_len), '--gpu-memory-utilization', str(CFG.gpu_mem_util),
           '--seed', '0', '--disable-log-requests']
    if 'AWQ' in modelo:
        cmd += ['--quantization', 'awq']
    else:
        cmd += ['--dtype', 'bfloat16']
    if tools:
        cmd += ['--enable-auto-tool-choice', '--tool-call-parser', 'llama3_json']
        t = plantilla_tools()
        if t:
            cmd += ['--chat-template', t]
    if loras:
        cmd += ['--enable-lora', '--max-lora-rank', str(CFG.lora_r), '--max-loras', str(min(len(loras), 4)), '--max-cpu-loras', str(len(loras)),
                '--lora-modules'] + [f'{n}={p}' for n, p in loras.items()]
    cmd += list(extra)
    log = open(f"{DIRS['logs']}/vllm_{int(time.time())}.log", 'w')
    SERVER['proc'] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env={**os.environ})
    SERVER['model'] = modelo
    for _ in range(180):
        time.sleep(5)
        with contextlib.suppress(Exception):
            if requests.get(f'http://localhost:{PORT}/health', timeout=2).ok:
                print('Servidor listo:', modelo, '| LoRA:', list((loras or {}).keys())); return
        if SERVER['proc'].poll() is not None:
            raise RuntimeError('vLLM terminó; revisa el log en ' + log.name)
    raise TimeoutError('vLLM no respondió')

def parar_servidor():
    if SERVER['proc'] is not None and SERVER['proc'].poll() is None:
        SERVER['proc'].terminate()
        with contextlib.suppress(Exception):
            SERVER['proc'].wait(60)
    SERVER['proc'] = None
    import gc; gc.collect()
    with contextlib.suppress(Exception):
        import torch; torch.cuda.empty_cache()

CLIENT = AsyncOpenAI(base_url=f'http://localhost:{PORT}/v1', api_key='local')

async def chat(model, msgs, tools=None, max_tokens=None, temperature=None):
    kw = dict(model=model, messages=msgs, temperature=CFG.temperatura if temperature is None else temperature,
              max_tokens=max_tokens or CFG.max_tokens_salida, seed=0)
    if tools:
        kw.update(tools=tools, tool_choice='auto')
    t0 = time.perf_counter()
    r = await CLIENT.chat.completions.create(**kw)
    return r, time.perf_counter() - t0

async def gather_limit(coros, n):
    sem = asyncio.Semaphore(n)
    async def run(c):
        async with sem:
            return await c
    return await asyncio.gather(*(run(c) for c in coros))

## 4. Datos sintéticos para el fine-tuning (GPU, ~2 h)
A partir **solo** de los fragmentos de K, el propio modelo base genera para cada fragmento 2 pares pregunta-respuesta (con el mismo formato JSON de salida) y 1 reformulación del fragmento. Después se descartan las preguntas cuya similitud con alguna pregunta evaluada supere el umbral (control de fuga).

In [ ]:
PROMPT_SINT = ('A partir del siguiente fragmento, genera un objeto JSON con dos campos: "reformulacion" (el mismo contenido '
               'reescrito con otras palabras, sin añadir información) y "qa" (una lista de {n} objetos con "pregunta", '
               '"respuesta" breve y "justificacion" de una o dos oraciones, todos respondibles solo con el fragmento).\n\n'
               'Título: {titulo}\nFragmento: {texto}')

async def _sint(fid):
    f = FRAG[fid]
    msgs = [{'role': 'user', 'content': PROMPT_SINT.format(n=CFG.qa_por_fragmento, titulo=f['titulo'], texto=f['texto'])}]
    r, _ = await chat(CFG.generador, msgs, max_tokens=700, temperature=0.7)
    txt = r.choices[0].message.content
    try:
        o = json.loads(re.findall(r'\{.*\}', txt, flags=re.S)[0])
    except Exception:
        o = {'reformulacion': '', 'qa': []}
    return {'fid': fid, 'bench': f['bench'], 'reformulacion': o.get('reformulacion', ''), 'qa': o.get('qa', [])[:CFG.qa_por_fragmento]}

async def generar_sinteticos():
    out = f"{DIRS['ft']}/sinteticos.jsonl"
    hechos = done_keys(out, lambda r: r['fid'])
    pend = [k for k in FRAG if k not in hechos]
    print('Fragmentos pendientes:', len(pend))
    for i in range(0, len(pend), 256):
        jl_append(out, await gather_limit([_sint(k) for k in pend[i:i + 256]], CFG.concurrencia * 2))

with etapa('4_sinteticos'):
    iniciar_servidor(CFG.generador)
    await generar_sinteticos()
    parar_servidor()

In [ ]:
# Filtro de fuga y construcción de los conjuntos de entrenamiento (uno por benchmark)
def construir_datasets_ft():
    emb = cargar_embedder()
    for b in BENCHS:
        dp = f"{DIRS['ft']}/train_{b}.jsonl"
        if os.path.exists(dp):
            print('Existe', dp); continue
        Q = np.load(f"{DIRS['indice']}/{b}_Q.npy")
        sint = [s for s in jl_read(f"{DIRS['ft']}/sinteticos.jsonl") if s['bench'] == b]
        qa = [(s['fid'], x) for s in sint for x in s['qa'] if isinstance(x, dict) and x.get('pregunta')]
        V = embed(emb, [x['pregunta'] for _, x in qa]) if qa else np.zeros((0, Q.shape[1]), 'float32')
        maxsim = (V @ Q.T).max(axis=1) if len(qa) else np.array([])
        textos, descartadas = [], 0
        for (fid, x), ms in zip(qa, maxsim):
            if ms > CFG.umbral_fuga:
                descartadas += 1; continue
            salida = json.dumps({'respuesta': x.get('respuesta', ''), 'justificacion': x.get('justificacion', '')}, ensure_ascii=False)
            msgs = mensajes('FT', x['pregunta']) + [{'role': 'assistant', 'content': salida}]
            textos.append({'text': TOK.apply_chat_template(msgs, tokenize=False)})
        for s in sint:
            f = FRAG[s['fid']]
            textos.append({'text': f"{f['titulo']}\n{f['texto']}"})
            if s['reformulacion']:
                textos.append({'text': f"{f['titulo']}\n{s['reformulacion']}"})
        random.Random(CFG.semilla_muestreo).shuffle(textos)
        jl_append(dp, textos)
        print(b, '| ejemplos:', len(textos), '| QA descartadas por fuga:', descartadas)

with etapa('4b_datasets_ft'):
    construir_datasets_ft()

## 5. Entrenamiento QLoRA (GPU, ~14 h en total)
Un adaptador por benchmark y semilla (6 entrenamientos). Cada entrenamiento guarda *checkpoints* en Drive y se reanuda automáticamente si la sesión se corta. **Ejecutar en una sesión sin servidor vLLM** (reiniciar el entorno antes). Se recomienda un entrenamiento por sesión.

In [ ]:
def entrenar(bench, semilla):
    import torch
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig
    nombre = f'ft_{bench}_s{semilla}'
    out = f"{DIRS['adaptadores']}/{nombre}"
    if os.path.exists(f'{out}/adapter_config.json'):
        print('Ya entrenado:', nombre); return
    set_seed(semilla)
    ds = Dataset.from_list(jl_read(f"{DIRS['ft']}/train_{bench}.jsonl"))
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(CFG.generador, quantization_config=bnb, torch_dtype=torch.bfloat16,
                                                 device_map='auto', token=os.environ.get('HF_TOKEN'))
    lora = LoraConfig(r=CFG.lora_r, lora_alpha=CFG.lora_alpha, lora_dropout=CFG.lora_dropout, bias='none', task_type='CAUSAL_LM',
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
    args = SFTConfig(output_dir=f'{out}_ckpt', num_train_epochs=CFG.epocas, learning_rate=CFG.lr, lr_scheduler_type='cosine',
                     warmup_ratio=0.03, per_device_train_batch_size=CFG.batch_size, gradient_accumulation_steps=CFG.grad_accum,
                     bf16=True, logging_steps=20, save_steps=200, save_total_limit=2, seed=semilla, data_seed=semilla,
                     gradient_checkpointing=True, report_to=['wandb'] if os.environ.get('WANDB_API_KEY') else [],
                     run_name=nombre, dataset_text_field='text', max_length=CFG.max_seq_len, packing=True)
    tr = SFTTrainer(model=model, args=args, train_dataset=ds, peft_config=lora, processing_class=TOK)
    ckpts = sorted(glob.glob(f'{out}_ckpt/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
    t0 = time.time()
    tr.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
    tr.model.save_pretrained(out)
    jl_append(f"{DIRS['logs']}/ft.jsonl", [{'adaptador': nombre, 'horas': (time.time() - t0) / 3600, 'ejemplos': len(ds),
                                            'vram_max_gb': torch.cuda.max_memory_allocated() / 1e9}])
    del tr, model; torch.cuda.empty_cache()

# Ejecutar uno a uno (uno por sesión es lo más seguro con Colab Pro):
BENCH_FT, SEMILLA_FT = 'hotpotqa', 13
with etapa(f'5_qlora_{BENCH_FT}_{SEMILLA_FT}'):
    entrenar(BENCH_FT, SEMILLA_FT)

## 6. Inferencia de las cinco configuraciones (GPU, ~4 h)
B0, RAG, FT, FC y oráculo, 3 repeticiones, temperatura 0. Resultados por fila en `salidas/inferencia.jsonl` (reanudable). Para FT, la repetición *r* usa el adaptador de la semilla *r*.

In [ ]:
def adaptadores_disponibles():
    out = {}
    for b in BENCHS:
        for s in CFG.semillas_ft:
            p = f"{DIRS['adaptadores']}/ft_{b}_s{s}"
            if os.path.exists(f'{p}/adapter_config.json'):
                out[f'ft_{b}_s{s}'] = p
    return out

async def responder(config, q, rep, retr=None):
    b = q['bench']; base = {'qid': q['qid'], 'bench': b, 'config': config, 'rep': rep}
    if config == 'FC':
        msgs = mensajes('FC', q['pregunta']); usados, llamadas, lat, tok_in, tok_out = [], [], 0.0, 0, 0
        invalidas = 0
        for paso in range(CFG.fc_max_llamadas + 1):
            ultimo = paso == CFG.fc_max_llamadas
            if ultimo:
                msgs.append({'role': 'user', 'content': 'Has alcanzado el límite de búsquedas. Responde ahora con el JSON final.'})
            r, t = await chat(CFG.generador, msgs, tools=None if ultimo else TOOL_BUSCAR)
            lat += t; tok_in += r.usage.prompt_tokens; tok_out += r.usage.completion_tokens
            m = r.choices[0].message
            if not m.tool_calls:
                break
            msgs.append({'role': 'assistant', 'content': m.content or '', 'tool_calls': [tc.model_dump() for tc in m.tool_calls[:1]]})
            tc = m.tool_calls[0]
            try:
                consulta = json.loads(tc.function.arguments).get('consulta', '')
            except Exception:
                consulta = ''; invalidas += 1
            t0 = time.perf_counter(); fids = retr.buscar(b, consulta, CFG.k_fc) if consulta else []; lat += time.perf_counter() - t0
            usados += [f for f in fids if f not in usados]; llamadas.append(consulta)
            msgs.append({'role': 'tool', 'tool_call_id': tc.id, 'content': fmt_fragmentos([FRAG[f] for f in fids]) or 'Sin resultados.'})
        out = parse_salida(m.content)
        return {**base, **out, 'contexto': usados, 'llamadas': llamadas, 'n_llamadas': len(llamadas), 'llamadas_invalidas': invalidas,
                'latencia_s': lat, 'tok_in': tok_in, 'tok_out': tok_out}
    if config == 'RAG':
        frs = RECUP[q['qid']]['rag']
    elif config == 'OR':
        frs = RECUP[q['qid']]['oraculo']
    else:
        frs = []
    model = f"ft_{b}_s{CFG.semillas_ft[rep]}" if config == 'FT' else CFG.generador
    t_ret = 0.0
    if config == 'RAG' and retr is not None:      # en la medición de latencia, RAG recupera en línea igual que FC
        t0 = time.perf_counter(); frs = retr.buscar(b, q['pregunta'], CFG.k_rag); t_ret = time.perf_counter() - t0
    r, t = await chat(model, mensajes(config, q['pregunta'], [FRAG[f] for f in frs]))
    return {**base, **parse_salida(r.choices[0].message.content), 'contexto': frs, 'latencia_s': t + t_ret,
            'tok_in': r.usage.prompt_tokens, 'tok_out': r.usage.completion_tokens}

async def inferencia(configs=CONFIGS, reps=CFG.repeticiones):
    out = f"{DIRS['salidas']}/inferencia.jsonl"
    hechos = done_keys(out, lambda r: (r['qid'], r['config'], r['rep']))
    retr = Recuperador('cpu') if 'FC' in configs else None
    ads = adaptadores_disponibles()
    for config in configs:
        for rep in reps:
            qs = [q for q in PREG.values() if (q['qid'], config, rep) not in hechos]
            if config == 'FT':
                qs = [q for q in qs if f"ft_{q['bench']}_s{CFG.semillas_ft[rep]}" in ads]
            if not qs:
                continue
            print(config, rep, len(qs))
            for i in range(0, len(qs), 200):
                rows = await gather_limit([responder(config, q, rep, retr if config == 'FC' else None) for q in qs[i:i + 200]], CFG.concurrencia)
                jl_append(out, rows)

with etapa('6_inferencia'):
    iniciar_servidor(CFG.generador, loras=adaptadores_disponibles(), tools=True)
    await inferencia()

## 7. Latencia sin lotes (GPU, ~1,5 h)
Submuestra fija de 100 preguntas por benchmark, procesadas **de una en una** (concurrencia 1), con la recuperación en línea en RAG y FC sobre el mismo recuperador en CPU. Ejecutar con el servidor de la etapa 6 activo.

In [ ]:
async def latencia():
    out = f"{DIRS['salidas']}/latencia.jsonl"
    hechos = done_keys(out, lambda r: (r['qid'], r['config']))
    retr = Recuperador('cpu'); rng = random.Random(CFG.semilla_muestreo)
    for b in BENCHS:
        qs = sorted([q for q in PREG.values() if q['bench'] == b], key=lambda q: q['qid'])
        sub = rng.sample(qs, min(CFG.n_latencia, len(qs)))
        for config in CONFIGS:
            for q in sub:
                if (q['qid'], config) in hechos:
                    continue
                r = await responder(config, q, 0, retr if config in ('RAG', 'FC') else None)
                jl_append(out, [{k: r[k] for k in ('qid', 'bench', 'config', 'latencia_s', 'tok_in', 'tok_out')}])

with etapa('7_latencia'):
    await latencia()
    parar_servidor()

## 8. Evaluación con el modelo juez (GPU, ~7 h)
Reiniciar el entorno antes (libera la memoria del generador). El juez Qwen2.5-14B-Instruct-AWQ valora: (a) en FRAMES, la equivalencia entre respuesta y referencia; (b) **F_ref**, afirmaciones respaldadas por la evidencia de referencia E\* (todas las configuraciones); (c) **F_ctx**, afirmaciones respaldadas por el contexto recibido (RAG, FC, OR). Las respuestas idénticas entre repeticiones se juzgan una sola vez (caché por *hash*).

In [ ]:
PROMPT_EQUIV = ('Pregunta: {pregunta}\nRespuesta de referencia: {ref}\nRespuesta evaluada: {pred}\n\n'
                '¿La respuesta evaluada es equivalente a la de referencia (mismo contenido, aunque cambie la forma)? '
                'Responde solo con JSON: {{"equivalente": true}} o {{"equivalente": false}}.')
PROMPT_FIEL = ('Evidencia:\n{evidencia}\n\nTexto a verificar:\nRespuesta: {resp}\nJustificación: {just}\n\n'
               'Divide el texto a verificar en afirmaciones atómicas y, para cada una, indica si la evidencia la respalda '
               '(true) o no (false). No uses conocimiento externo. Responde solo con JSON: '
               '{{"afirmaciones": [{{"texto": "...", "respaldada": true}}]}}')

_EV = {}
def evidencia_ref(q, r, emb=None):
    if q['bench'] == 'hotpotqa':
        return evidencia_ref_hotpot(q)
    # FRAMES: fragmentos de los artículos de referencia más similares al texto evaluado
    if 'fids' not in _EV:
        _EV['fids'] = json.load(open(f"{DIRS['indice']}/frames_fids.json")); _EV['X'] = np.load(f"{DIRS['indice']}/frames_X.npy")
        _EV['url'] = {}
        for j, f in enumerate(_EV['fids']):
            _EV['url'].setdefault(FRAG[f]['url'], []).append(j)
    gold = [j for u in q['wiki_links'] for j in _EV['url'].get(u, [])]
    if not gold:
        return []
    v = embed(emb, [f"{r['respuesta']} {r['justificacion']}"])[0]
    s = _EV['X'][gold] @ v
    return [FRAG[_EV['fids'][gold[j]]] for j in np.argsort(-s)[:CFG.k_verificacion_frames]]

async def juzgar(tarea, prompt):
    r, _ = await chat(CFG.juez, [{'role': 'user', 'content': prompt}], max_tokens=600)
    txt = r.choices[0].message.content
    try:
        return json.loads(re.findall(r'\{.*\}', txt, flags=re.S)[0]), True
    except Exception:
        return {}, False

def puntuacion_fidelidad(o):
    af = o.get('afirmaciones') or []
    af = [a for a in af if isinstance(a, dict) and 'respaldada' in a]
    return (sum(bool(a['respaldada']) for a in af) / len(af)) if af else None

async def evaluar_juez():
    inf = jl_read(f"{DIRS['salidas']}/inferencia.jsonl")
    cache_p = f"{DIRS['juez']}/cache.jsonl"
    cache = {c['h']: c for c in jl_read(cache_p)}
    emb = cargar_embedder('cpu')
    mapa_p = f"{DIRS['juez']}/mapa.jsonl"
    mapa = {(m['qid'], m['config'], m['rep']): m for m in jl_read(mapa_p)}
    tareas, nuevos = {}, []
    for r in inf:
        q = PREG[r['qid']]; key = (r['qid'], r['config'], r['rep'])
        m = mapa.get(key)
        if m is None:
            m = {'qid': r['qid'], 'config': r['config'], 'rep': r['rep']}
            ev = evidencia_ref(q, r, emb)
            m['ev_ref'] = [e['texto'] for e in ev] if q['bench'] == 'hotpotqa' else [e['fid'] for e in ev]
            if q['bench'] == 'frames':
                m['equiv'] = sha('equiv', q['qid'], r['respuesta'])
            m['fref'] = sha('fref', q['qid'], r['respuesta'], r['justificacion'], *m['ev_ref'])
            if r['config'] in ('RAG', 'FC', 'OR'):
                m['fctx'] = sha('fctx', q['qid'], r['respuesta'], r['justificacion'], *r['contexto'])
            nuevos.append(m)
        ev = ([{'titulo': '', 'texto': t} for t in m['ev_ref']] if q['bench'] == 'hotpotqa' else [FRAG[f] for f in m['ev_ref']])
        if 'equiv' in m:
            tareas[m['equiv']] = ('equiv', PROMPT_EQUIV.format(pregunta=q['pregunta'], ref=q['respuesta'], pred=r['respuesta']))
        tareas[m['fref']] = ('fref', PROMPT_FIEL.format(evidencia=fmt_fragmentos(ev), resp=r['respuesta'], just=r['justificacion']))
        if 'fctx' in m:
            ctx = [FRAG[f] for f in r['contexto']]
            tareas[m['fctx']] = ('fctx', PROMPT_FIEL.format(evidencia=fmt_fragmentos(ctx) or '(sin fragmentos)', resp=r['respuesta'], just=r['justificacion']))
    jl_append(mapa_p, nuevos)
    pend = [(h, t) for h, t in tareas.items() if h not in cache]
    print(f'Juicios únicos: {len(tareas)} | pendientes: {len(pend)} | respuestas: {len(inf)}')
    for i in range(0, len(pend), 200):
        res = await gather_limit([juzgar(t, p) for _, (t, p) in pend[i:i + 200]], CFG.concurrencia)
        jl_append(cache_p, [{'h': h, 'tarea': t[0], 'salida': o, 'ok': ok} for (h, t), (o, ok) in zip(pend[i:i + 200], res)])

with etapa('8_juez'):
    iniciar_servidor(CFG.juez)
    await evaluar_juez()
    parar_servidor()

## 9. Métricas, validación del juez y análisis estadístico (CPU)
No requiere GPU: conviene ejecutarlo con un entorno solo CPU para no consumir unidades.

In [ ]:
import pandas as pd

def tabla_resultados():
    inf = jl_read(f"{DIRS['salidas']}/inferencia.jsonl")
    cache = {c['h']: c for c in jl_read(f"{DIRS['juez']}/cache.jsonl")}
    mapa = {(m['qid'], m['config'], m['rep']): m for m in jl_read(f"{DIRS['juez']}/mapa.jsonl")}
    filas = []
    for r in inf:
        q = PREG[r['qid']]; f = {**{k: r.get(k) for k in ('qid', 'bench', 'config', 'rep', 'latencia_s', 'tok_in', 'tok_out', 'formato_valido', 'n_llamadas')}}
        f['em'] = exact_match(r['respuesta'], q['respuesta']); f['f1'] = f1_score(r['respuesta'], q['respuesta'])
        m = mapa.get((r['qid'], r['config'], r['rep']), {})
        if q['bench'] == 'frames':
            c = cache.get(m.get('equiv'))
            f['acc'] = float(bool(c['salida'].get('equivalente'))) if c and c['ok'] else np.nan
        else:
            f['acc'] = f['em']
        c = cache.get(m.get('fref'))
        f['f_ref'] = puntuacion_fidelidad(c['salida']) if c else np.nan
        c = cache.get(m.get('fctx'))
        f['f_ctx'] = puntuacion_fidelidad(c['salida']) if c else np.nan
        filas.append(f)
    df = pd.DataFrame(filas)
    df['f_ref'] = pd.to_numeric(df['f_ref']); df['alucinacion'] = 1 - df['f_ref']
    df.to_csv(f"{DIRS['analisis']}/por_respuesta.csv", index=False)
    return df

In [ ]:
# 9.1 Validación del juez: exportar muestra para anotación manual y calcular kappa
from sklearn.metrics import cohen_kappa_score

def exportar_anotacion(df):
    p = f"{DIRS['analisis']}/anotacion_manual.csv"
    if os.path.exists(p):
        print('Ya exportado:', p); return
    inf = {(r['qid'], r['config'], r['rep']): r for r in jl_read(f"{DIRS['salidas']}/inferencia.jsonl")}
    mapa = {(m['qid'], m['config'], m['rep']): m for m in jl_read(f"{DIRS['juez']}/mapa.jsonl")}
    m = df[df['rep'] == 0].groupby('config', group_keys=False).apply(
        lambda g: g.sample(min(len(g), CFG.n_anotacion // len(CONFIGS)), random_state=CFG.semilla_muestreo))
    rows = []
    for _, x in m.iterrows():
        r = inf[(x['qid'], x['config'], 0)]; q = PREG[x['qid']]
        ev_ids = mapa[(x['qid'], x['config'], 0)]['ev_ref']
        ev = [{'titulo': '', 'texto': t} for t in ev_ids] if q['bench'] == 'hotpotqa' else [FRAG[f] for f in ev_ids]
        rows.append({'qid': x['qid'], 'config': x['config'], 'pregunta': q['pregunta'], 'respuesta': r['respuesta'],
                     'justificacion': r['justificacion'], 'evidencia': fmt_fragmentos(ev),
                     'humano_todo_respaldado(0/1)': ''})
    pd.DataFrame(rows).sample(frac=1, random_state=1).to_csv(p, index=False)   # sin la nota del juez, en orden aleatorio
    print('Anota la última columna y guarda el archivo con el mismo nombre:', p)

def kappa_juez(df):
    a = pd.read_csv(f"{DIRS['analisis']}/anotacion_manual.csv")
    a = a[a['humano_todo_respaldado(0/1)'].astype(str).isin(['0', '1'])]
    j = df[df['rep'] == 0].set_index(['qid', 'config'])['f_ref']
    juez = [int(j.loc[(r.qid, r.config)] == 1.0) for r in a.itertuples()]
    k = cohen_kappa_score(a['humano_todo_respaldado(0/1)'].astype(int), juez)
    print(f'kappa = {k:.3f} (n = {len(a)}) → {"ACEPTADO" if k >= CFG.kappa_minimo else "REVISAR instrucciones del juez"}')
    return k

In [ ]:
# 9.2 Pruebas pareadas: McNemar (binarias, voto mayoritario de las 3 repeticiones), bootstrap pareado (continuas),
#     Wilcoxon (latencia) y corrección de Holm-Bonferroni.
from itertools import combinations
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

BINARIAS = ['acc', 'em']
CONTINUAS = ['f1', 'f_ref', 'f_ctx']

def por_pregunta(df, metrica):
    g = df.groupby(['qid', 'config'])[metrica]
    if metrica in BINARIAS:
        return (g.mean() > 0.5).astype(float).unstack()     # valor mayoritario
    return g.mean().unstack()                               # media de repeticiones

def bootstrap_pareado(a, b, n=None, seed=0):
    n = n or CFG.bootstrap; d = np.asarray(a) - np.asarray(b); rng = np.random.default_rng(seed)
    bs = rng.choice(d, size=(n, len(d)), replace=True).mean(axis=1)
    p = 2 * min((bs <= 0).mean(), (bs >= 0).mean())
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), min(1.0, p)

def comparar(df, bench, metrica, configs=CONFIGS):
    t = por_pregunta(df[df['bench'] == bench], metrica)
    filas = []
    for a, b in combinations([c for c in configs if c in t.columns], 2):
        x = t[[a, b]].dropna()
        if len(x) < 10:
            continue
        if metrica in BINARIAS:
            n01 = int(((x[a] == 0) & (x[b] == 1)).sum()); n10 = int(((x[a] == 1) & (x[b] == 0)).sum())
            tabla = [[int(((x[a] == 1) & (x[b] == 1)).sum()), n10], [n01, int(((x[a] == 0) & (x[b] == 0)).sum())]]
            p = mcnemar(tabla, exact=(n01 + n10) < 25, correction=True).pvalue
            d, lo, hi, _ = bootstrap_pareado(x[a], x[b]); prueba = 'McNemar'
        else:
            d, lo, hi, p = bootstrap_pareado(x[a], x[b]); prueba = 'bootstrap'
        filas.append({'bench': bench, 'metrica': metrica, 'A': a, 'B': b, 'n': len(x), 'dif_A_menos_B': d, 'ic95_inf': lo, 'ic95_sup': hi, 'p': p, 'prueba': prueba})
    out = pd.DataFrame(filas)
    if len(out):
        out['p_holm'] = multipletests(out['p'], method='holm')[1]
    return out

def comparar_latencia(lat, bench):
    t = lat[lat['bench'] == bench].pivot_table(index='qid', columns='config', values='latencia_s')
    filas = []
    for a, b in combinations([c for c in CONFIGS if c in t.columns], 2):
        x = t[[a, b]].dropna()
        p = wilcoxon(x[a], x[b]).pvalue if len(x) >= 10 else np.nan
        filas.append({'bench': bench, 'metrica': 'latencia', 'A': a, 'B': b, 'n': len(x), 'mediana_A': x[a].median(), 'mediana_B': x[b].median(), 'p': p})
    out = pd.DataFrame(filas)
    if len(out):
        out['p_holm'] = multipletests(out['p'].fillna(1), method='holm')[1]
    return out

In [ ]:
# 9.3 Resumen por configuración (media ± desviación entre repeticiones), coste y dominancia de Pareto
def resumen(df, lat=None):
    m = ['acc', 'em', 'f1', 'f_ref', 'f_ctx', 'alucinacion']
    por_rep = df.groupby(['bench', 'config', 'rep'])[[c for c in m if c in df]].mean()
    res = por_rep.groupby(['bench', 'config']).agg(['mean', 'std'])
    if lat is not None and len(lat):
        l = lat.groupby(['bench', 'config'])['latencia_s'].agg(mediana='median', p90=lambda s: s.quantile(0.9))
        res = res.join(pd.concat({'latencia_s': l}, axis=1))
    return res

def coste_por_1000(c_fijo_uc, c_var_uc_por_pregunta, N):
    # C(N) = C_fijo / N × 1000 + c_var × 1000  (apartado 4.4.3), en unidades de cómputo
    return c_fijo_uc / N * 1000 + c_var_uc_por_pregunta * 1000

def dominancia(medias, pruebas, mayor_mejor, sin_prueba=('coste',)):
    # medias: DataFrame index=config, columnas=métricas (valores medios)
    # pruebas: dict (A, B, metrica) -> p_holm ; mayor_mejor: dict metrica -> True si más es mejor
    # Criterio (4.5): A es mejor que B en una métrica si la diferencia es significativa (p_holm < 0,05) y supera el
    # umbral de relevancia práctica: 5 pp en precisión/fidelidad; 20 % en latencia/coste (el coste es determinista).
    def p(a, b, met):
        if met in sin_prueba:
            return 0.0
        return pruebas.get((a, b, met), pruebas.get((b, a, met), 1.0))
    def mejor(a, b, met):
        va, vb = medias.loc[a, met], medias.loc[b, met]
        if mayor_mejor[met]:
            return p(a, b, met) < 0.05 and (va - vb) >= CFG.umbral_pp - 1e-9
        return p(a, b, met) < 0.05 and vb > 0 and (vb - va) / vb >= CFG.umbral_rel_coste - 1e-9
    cfgs = list(medias.index); dominadas = set()
    for a in cfgs:
        for b in cfgs:
            if a != b and not any(mejor(b, a, m) for m in mayor_mejor) and any(mejor(a, b, m) for m in mayor_mejor):
                dominadas.add(b)
    return [c for c in cfgs if c not in dominadas], sorted(dominadas)

In [ ]:
# 9.4 Ejecutar el análisis
with etapa('9_analisis'):
    DF = tabla_resultados()
    exportar_anotacion(DF)
    LAT = pd.DataFrame(jl_read(f"{DIRS['salidas']}/latencia.jsonl"))
    RES = resumen(DF, LAT); RES.to_csv(f"{DIRS['analisis']}/resumen.csv"); display(RES.round(3))
    PRU = pd.concat([comparar(DF, b, m) for b in BENCHS for m in ['acc', 'f1', 'f_ref', 'f_ctx']] +
                    [comparar_latencia(LAT, b) for b in BENCHS if len(LAT)], ignore_index=True)
    PRU.to_csv(f"{DIRS['analisis']}/pruebas.csv", index=False); display(PRU.round(4))
    # Indicadores de comportamiento de FC
    fc = DF[DF['config'] == 'FC']
    if len(fc):
        display(fc.groupby('bench').agg(llamadas_media=('n_llamadas', 'mean'), sin_busqueda=('n_llamadas', lambda s: (s == 0).mean()),
                                        formato_valido=('formato_valido', 'mean')))
    # kappa (cuando el archivo esté anotado)
    with contextlib.suppress(Exception):
        kappa_juez(DF)
    # Presupuesto consumido
    display(pd.DataFrame(jl_read(f"{DIRS['logs']}/budget.jsonl")).groupby('etapa')[['horas', 'uc_estimadas']].sum().round(2))

## 10. Bloque complementario C2: StableToolBench (opcional, GPU, ~5 h)
Se ejecuta solo si el presupuesto lo permite (regla de priorización del apartado 3.3). Requiere descargar del repositorio de StableToolBench las consultas resolubles, la documentación de herramientas y la **caché** de respuestas de API, y desplegar o apuntar a su servidor virtual. Este bloque deja preparada la estructura y las tres variantes (FC, FC+RAG sobre la documentación de API, FC+FT); la función de llamada al servidor debe completarse con el formato exacto del README del benchmark.

In [ ]:
STB_DIR = f'{CFG.drive_root}/stabletoolbench'      # consultas resolubles, herramientas y caché descargadas aquí
STB_SERVER = 'http://localhost:8080/virtual'      # servidor virtual de StableToolBench (ver su README)
N_C2 = 150

def cargar_consultas_stb():
    # Espera archivos JSON de los subconjuntos resolubles (p. ej. G1_instruction.json, ...) en STB_DIR/solvable_queries
    qs = []
    for p in sorted(glob.glob(f'{STB_DIR}/solvable_queries/*.json')):
        for x in json.load(open(p)):
            qs.append({'grupo': os.path.basename(p)[:-5], **x})
    rng = random.Random(CFG.semilla_muestreo); rng.shuffle(qs)
    return qs[:N_C2]

def llamar_api_stb(categoria, herramienta, api, argumentos):
    # Completar según el README de StableToolBench (petición al servidor virtual con caché).
    # Mantener 'solo caché': si la llamada no está en la caché publicada, devolver un error controlado
    # y excluir la consulta del cómputo, documentando cuántas se excluyen.
    raise NotImplementedError('Adaptar al protocolo del servidor de StableToolBench antes de ejecutar C2')

VARIANTES_C2 = ['FC', 'FC+RAG', 'FC+FT']
# El bucle de agente reutiliza chat() y el formato de herramientas de la etapa 6; en FC+RAG se recuperan antes
# las 5 descripciones de API más similares a la consulta (BGE-M3 sobre la documentación) y solo esas se exponen como
# herramientas; en FC+FT se sirve un adaptador QLoRA entrenado con trayectorias de ToolBench que no se solapan con N_C2.

## Registro de cambios y notas
* Cualquier cambio de un valor de `Config` tras el piloto debe anotarse aquí y en la memoria (apartado 4.3), aplicándose a todas las configuraciones afectadas.
* Tras el piloto, fijar las versiones de los paquetes de la celda 0.2 con las que figuren en `logs/manifiesto.jsonl`.